# Донавчання `best2.pt` для analog / digital / thermal

Ноутбук донавчає на підготовленому датасеті `best2_thermal_v1`. Навчання йде на GPU Google Colab; твій комп'ютер може бути вимкнений після запуску. Датасет і ваги мають лежати на Google Drive, а контрольні точки зберігаються в Drive після кожної епохи.

## Підготуй Google Drive
Створи `MyDrive/best2_training/input/` і поклади туди `best2.pt` та `best2_thermal_v1.zip`. ZIP має містити папку `best2_thermal_v1/` з `train/`, `val/`, `test/`; кожен split має підпапки `analog`, `digital`, `thermal_analog`, `thermal_digital`. На Mac створи архів командою з теки проєкту: `cd datasets && zip -qr /private/tmp/best2_thermal_v1.zip best2_thermal_v1`, потім завантаж цей ZIP і `best2.pt` у Drive.

Не перейменовуй класи. Не додавай кадри одного відео одночасно в різні split-и.

## Важливо про переривання
Colab виконує код на віддаленій віртуальній машині. Втрата інтернету на твоєму комп'ютері сама по собі не вимикає вже запущене навчання, і комп'ютер не мусить бути увімкнений. Але Colab може зупинити VM через ліміт часу або пам'яті. Якщо сесія обірветься, перевір наявність `last.pt`, запусти клітинки підготовки та постав `RESUME = True`. Ноутбук продовжить з ваг останнього checkpoint у новому запуску з безпечними налаштуваннями; стан оптимізатора почнеться заново.

На безкоштовному тарифі Google вказує максимум до 12 годин для одного ноутбука, але фактична доступність GPU та ліміти змінюються. Тому збереження в Drive обов'язкове.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, zipfile, os

ROOT = Path('/content/drive/MyDrive/best2_training')
INPUT = ROOT / 'input'
DATA_ZIP = INPUT / 'best2_thermal_v1.zip'
BASE_MODEL = INPUT / 'best2.pt'
DATASET = Path('/content/best2_thermal_v1')
RUNS = ROOT / 'runs'
RUN_NAME = 'thermal_finetune_v1'
RESUME = False  # Зміни на True лише щоб продовжити перерване навчання

assert BASE_MODEL.is_file(), f'Не знайдено ваги: {BASE_MODEL}'
assert DATA_ZIP.is_file(), f'Не знайдено архів датасету: {DATA_ZIP}'
if not DATASET.exists():
    with zipfile.ZipFile(DATA_ZIP) as archive:
        archive.extractall('/content')
assert (DATASET / 'train').is_dir() and (DATASET / 'val').is_dir() and (DATASET / 'test').is_dir()
print('Input files ready:', BASE_MODEL, DATASET)

In [ ]:
%pip -q install ultralytics scikit-learn

In [ ]:
import torch, ultralytics
from ultralytics import YOLO

print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не активний. У Colab відкрий Runtime → Change runtime type → GPU і запусти клітинки знову.')

expected = {'analog', 'digital', 'thermal_analog', 'thermal_digital'}
for split in ('train', 'val', 'test'):
    counts = {p.name: sum(1 for f in p.iterdir() if f.is_file()) for p in (DATASET / split).iterdir() if p.is_dir()}
    assert set(counts) == expected, f'{split}: неправильні теки класів: {counts}'
    print(split, counts)

base = YOLO(str(BASE_MODEL))
print('Model task:', base.task, '| class names:', base.names)
assert base.task == 'classify', 'best2.pt не класифікатор; зупинено до початку тренування.'
expected_order = sorted(expected)  # аналогічно порядку тек у classification dataset
assert [str(base.names[i]) for i in range(len(base.names))] == expected_order, 'Порядок класів моделі не збігається з датасетом; зупинено, щоб уникнути помилкових міток.'
print('GPU/model/dataset checks passed.')

## Базова оцінка
Це вимірює поточну якість `best2.pt` на validation-частині. Test split лишається недоторканим до фінальної перевірки, щоб чесно порівняти результат після навчання.

In [ ]:
baseline = base.val(data=str(DATASET), split='val', imgsz=224, batch=32, plots=True, project=str(RUNS), name='baseline_val', exist_ok=True)
print('Baseline validation top-1:', baseline.top1)
print('Confusion matrix and plots:', RUNS / 'baseline_val')

## Донавчання
`imgsz=224` відповідає типовому розміру для класифікаторів. `batch=-1` підбирає розмір під доступну GPU-пам'ять; якщо з'явиться CUDA out of memory, зміни його на `32` або `16`. Навчання перевіряє `val`, зупиняється раніше при плато, а `save_period=1` зберігає епохи.

In [ ]:
RUN_NAME = 'thermal_finetune_v1_safe'  # окремий запуск зі зменшеним споживанням пам'яті
if RESUME:
    candidates = [
        RUNS / 'thermal_finetune_v1_safe' / 'weights' / 'last.pt',
        RUNS / 'thermal_finetune_v1' / 'weights' / 'last.pt',
        RUNS / 'thermal_finetune_v1' / 'weights' / 'best.pt',
    ]
    start_weights = next((p for p in candidates if p.is_file()), None)
    assert start_weights is not None, f'Не знайдено checkpoint для відновлення. Перевір: {candidates}'
    trainer = YOLO(str(start_weights))  # ваги зберігаються; оптимізатор стартує заново
else:
    trainer = YOLO(str(BASE_MODEL))

train_result = trainer.train(
    data=str(DATASET),
    epochs=80,
    patience=12,
    imgsz=224,
    batch=32,
    device=0,
    workers=0,
    cache=False,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    save=True,
    save_period=1,
    val=True,
    plots=True,
    amp=True,
    seed=42,
    deterministic=True,
    project=str(RUNS),
    name=RUN_NAME,
    exist_ok=True,
)

BEST = RUNS / RUN_NAME / 'weights' / 'best.pt'
LAST = RUNS / RUN_NAME / 'weights' / 'last.pt'
assert BEST.is_file(), f'Не знайдено найкращі ваги: {BEST}'
print('Best checkpoint:', BEST)
print('Recovery checkpoint:', LAST)

## Фінальна перевірка
Після завершення навчаємося оцінювати на test лише найкращу модель, яку вибрала validation-частина. Перевір top-1 і confusion matrix: особливо дивись на помилки між чотирма класами. Якщо результат не кращий за baseline, не замінюй робочу модель.

In [ ]:
final_model = YOLO(str(BEST))
test = final_model.val(data=str(DATASET), split='test', imgsz=224, batch=32, plots=True, project=str(RUNS), name='thermal_finetune_v1_test', exist_ok=True)
base_model = YOLO(str(BASE_MODEL))
baseline_test = base_model.val(data=str(DATASET), split='test', imgsz=224, batch=32, plots=True, project=str(RUNS), name='baseline_test', exist_ok=True)
print('Fine-tuned test top-1:', test.top1)
print('Original best2 test top-1:', baseline_test.top1)
print('Test-set improvement (percentage points):', (test.top1 - baseline_test.top1) * 100)
print('Review confusion matrix and sample errors in:', RUNS / 'thermal_finetune_v1_test')
print('Use best.pt in the app only after checking its four class names and comparing real videos.')